# D1-S2-Nb3: The Powers of Low-Level Probing
Showcase:
- state-verification
- auto-correction / self-healing aPs

## Set your `protocol_mode`

...so it is easy to switch between simulation and execution

In [ ]:
prep_simulation = False  # True or False

star_simulation = True  # True or False

run_identifier = "nb3"

## Library Import

In [ ]:
import asyncio
import logging
import time

---
## Visualizer3D
...so everything that follows has one highest-level reference frame

In [ ]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

facility = Facility(name="facility", size_x=2_400, size_y=1_000, size_z=0)

viewer = Viewer3D(root = facility, name="nb3_probing")

await viewer.start()

---

## Set your `logging`
We (and our agents) always benefit from a log.

This follows standard Python logging but for convenience we set up:
- showing logging inside notebook using `verbose` declaration
- simple way to log to a file (in append mode)

I recommend always logging all commands you send/receive from any device for auditing purposes.

In [ ]:
import datetime
import logging
import time

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

# one mode per device: 4 combinations, each with its own log folder
prep_mode = "simulation" if prep_simulation else "execution"
star_mode = "simulation" if star_simulation else "execution"
protocol_mode = f"prep_{prep_mode}-star_{star_mode}"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)

pylabrobot.verbose(True, level=logging.INFO)

print(f"logging to {log_path}")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

## Device setups
### (shadow) STAR(let)

In [ ]:
from pylabrobot.hamilton.star import STARlet

star = STARlet(simulation=star_simulation)

facility.assign_child_resource(
    resource=star,
    location=Coordinate(x=800.0, y=0.0, z=0.0)
)

await star.setup()

### Prep

In [ ]:
from pylabrobot.hamilton.prep import Prep

HOST = "192.168.100.102"  # ignored in simulation
PORT = 2000

prep = Prep(
    simulation=prep_simulation,
    host=HOST,
    port=PORT,
    )

facility.assign_child_resource(
    prep,
    location=Coordinate(x=0.0, y=0.0, z=0.0)
)

await prep.setup()
await prep.lights.turn_on()

---
## Pickup probing tools
...all Hamilton liquid handling workstations come with "teaching needles" / metal tips attached to some part of the deck (usually the waste block)

you can easily find them by using PLR's `get_resource()` finding function:

In [ ]:
prep_tipspot = prep.get_resource("Prep_teaching_needle")
prep_tipspot

On the STAR there is not just *one* teaching needle, but an entire tiprack of teaching needles:

In [ ]:
star_teaching_tiprack = star.get_resource("STARlet_deck_teaching_tip_rack")
star_teaching_tiprack

Having the tipspots/tiprack readily available we can pick them up:

In [ ]:
prep_channel_idx = 1 # easier to see for you guys in the audience ;)

await prep.pipettes.pick_up_tips(
    tip_spots=[prep_tipspot], # has to be a list
    use_channels=[prep_channel_idx]
)

In [ ]:
# prep.get_resource("Prep_calibration_block").get_location_wrt(prep.deck,"c","c","t")

In [ ]:
# await prep.pipettes.probe_calibration_block_cct(channel_idx=1)

Since, even on the Prep, there are multiple pipettes - and we *could* therefore pick up multiple needles/tips - arguments always take a list.

The index of an element in the list declares which pipette it addresses.

N.B.: Python uses 0-based indexing! And therefore so does PyLabRobot.

N.B.2: Physical pipettes are counted from back to front, starting at 0!

In [ ]:
star_channel_idxs = [6, 7] # easier to see for you guys in the audience ;)

await star.pipettes.pick_up_tips(
    tip_spots=star_teaching_tiprack["A1:B1"],
    use_channels=star_channel_idxs
)

---
## Probing

### ... the Deck

move pipettes to the position you want to probe, then call ztouch probing method:

#### on Prep
The Prep has a handy little calibration block at a known position for calibration purposes:


In [ ]:
calibration_block = prep.get_resource("Prep_calibration_block")
calibration_block

To `get_` its center_x, center_y, top position we can use PLR's resource model:

In [ ]:
calibration_block_cct = calibration_block.get_location_wrt(
        other=prep.deck, # we always move motors relative to the deck
        x="c",
        y="c",
        z="t"
    )

calibration_block_cct

In [ ]:
await prep.pipettes.move_to_xy_positions(
    x = calibration_block_cct.x,
    ys = {prep_channel_idx : calibration_block_cct.y}
)

In [ ]:
measured_z = await prep.pipettes.probe_z_using_ztouch(
    channel_idx=prep_channel_idx,
    search_start_position = 110,
    # search_end_position: Optional[float] = None,
    # search_speed: float = 10.0,
    # push_force_pwm: Optional[int] = None,
    # end_tolerance: float = 1.2,
    # tip_len: Optional[float] = None,
    # allow_without_tip: bool = False,
    # post_detection_distance: float = 2.0,
    # move_to_safe_z_position_after: bool = False,
)
print(f"Prep: {measured_z=}")

await prep.pipettes.move_to_safe_z()

#### (Almost) the same for STAR:

In [ ]:
chosen_deck_coordinates = [
    Coordinate(400.0, 500.0, 245.0),
    Coordinate(400.0, 200.0, 245.0),
]
chosen_ys = [coord.y for coord in chosen_deck_coordinates]


await star.pipettes.move_to_xy_positions(
    x = chosen_deck_coordinates[0].x,
    ys = dict(zip(star_channel_idxs, chosen_ys))
)


measured_zs = await asyncio.gather(
    *(
        star.pipettes.probe_z_using_ztouch(
            channel=ch
        )
        for ch in star_channel_idxs)
  )
print(f"STARlet: {measured_zs=}")

await star.pipettes.move_to_safe_z()

### ... in multiple locations 

In [ ]:
await prep.pipettes.probe_deck_corner_z_offsets()

then we want to ensure we place the teaching needle back:

In [ ]:
# via an explicit drop command:

# await prep.pipettes.drop_tips(
#     tip_spots=[prep_tipspot], # has to be a list
#     use_channels=[prep_channel_idx]
# )

# or the lazy way :P

await prep.pipettes.return_tips()

The PLR STAR(let) does not yet have this deck probing convience method.

...but we still have to return the teaching needles on the STARlets:

In [ ]:
await star.pipettes.return_tips()

---
### ... tip presence

using the sleeve sensors of the pipettes we can check whether a tip exists at a tipspot.

i.e. we can verify the state of the resource model
...and correct it if the model is outdated!

But first, we need some tips and activate `set_tip_tracking`:

In [ ]:
from pylabrobot.resources import set_tip_tracking
from pylabrobot.resources.hamilton import (
    hamilton_96_tiprack_50uL_NTR,
    hamilton_96_tiprack_300uL_NTR
)

set_tip_tracking(enabled=True)

then create new tipracks and assign them to the deck:

In [ ]:
ntr_50ul_00 = prep.deck[1] = hamilton_96_tiprack_50uL_NTR(
    name="ntr_50ul_00",
    )

ntr_300ul_00 = prep.deck[7] = hamilton_96_tiprack_300uL_NTR(
    name="ntr_300ul_00",
    )

In [ ]:
start_time = time.time()

tip_probing_results = await prep.pipettes.probe_tip_presence_via_pickup(
    tip_spots=ntr_50ul_00["G4:H4"]
)

end_time = time.time() - start_time

tip_probing_results, f"duraction: {end_time} sec"

#### Update if model is outdated

In [ ]:
tip_probing_results = await prep.pipettes.probe_tip_presence_via_pickup(
    tip_spots=ntr_50ul_00["G1:H1"]
)

for spot_name, has_tip in tip_probing_results.items():
    print(spot_name, has_tip)

    if not has_tip:
        # the model said there was a tip,
        # the device found none:
        print(f" -> correct the model's tipstate")
        ntr_50ul_00.get_resource(spot_name).unassign_tip()

same on the STAR:

In [ ]:
from pylabrobot.lib.liquid_handling.tip_presence_probing import probe_tip_presence_via_pickup
from pylabrobot.resources.hamilton import hamilton_tip_carrier_L5_ntr_a00


tip_carrier = hamilton_tip_carrier_L5_ntr_a00(name="star_tip_carrier")
tip_carrier[0] = star_ntr_50ul_00 = hamilton_96_tiprack_50uL_NTR(name="star_ntr_50ul_00")
star.deck.assign_child_resource(tip_carrier, track=13)

In [ ]:
# the STAR has no `probe_tip_presence_via_pickup` of its own yet: the shared helper drives its calls
tip_probing_results = await probe_tip_presence_via_pickup(
    tip_spots=star_ntr_50ul_00["G1:H1"],
    use_channels=star_channel_idxs,
    pick_up_tips=lambda spots, channels: star.pipettes.pick_up_tips(spots, use_channels=channels),
    drop_tips=lambda spots, channels: star.pipettes.drop_tips(spots, use_channels=channels),
    sense_tip_presence=star.pipettes.sense_tip_presence,
)

for spot_name, has_tip in tip_probing_results.items():
    print(spot_name, has_tip)

    if not has_tip:
        print(f" -> correct the model's tipstate")
        star_ntr_50ul_00.get_resource(spot_name).unassign_tip()

## STAR only: carrier sensing

In [ ]:
await star.autoload.sense_carrier_presence_on_deck()

In [ ]:
await star.autoload.sense_carrier_presence_on_loading_tray()
# N.B.: this will move the autoload back and forth
# along the entire x-length of the device

In [ ]:
for track_idx in [40, 39, 41]:
    sense_result = await star.autoload.sense_carrier_presence_on_single_loading_tray_track(
        track=track_idx,
        park_after=False
    )
    print(f"{track_idx}: track present? -> {sense_result}")

await star.autoload.park()

---
# Shutdown

In [ ]:
await star.stop()

In [ ]:
await prep.stop()

In [ ]:
await viewer.stop()